In [1]:
import importlib

import pandas as pd
import numpy as np

import attrition_analysis.data_selection as data_selection
import attrition_analysis.models_utils as models_utils

importlib.reload(data_selection)
importlib.reload(models_utils)

from attrition_analysis.models_utils import (
    split_train_test_df,
    estimators_dict,
    param_distributions_dict,
    categorical_models_dict_c,
    mixed_models_dict_c,
    run_logistic_cross_validation,
    run_logistic_gap_analysis,
    run_logistic_model_comparison,
    tune_logistic_hyperparameters_top_combinations,
    evaluate_thresholds_optimized_logistic_models_cv
)


df = pd.read_csv("../../data/clean/Employee-Attrition_Clean.csv")

target = "AttritionFlag"

df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,EducationField,EmployeeCount,EmployeeNumber,Gender,...,IncomeGroup,EducationLevel,StockOption,JobLevelGroup,SatisfactionLevel,PerformanceRatingLevel,EnvironmentSatisfactionLevel,JobInvolvementLevel,RelationshipSatisfactionLevel,WorkLifeBalanceLevel
0,41,Yes,Travel_Rarely,1102,Sales,1,Life Sciences,1,1,Female,...,Medium,College,No Options,Junior Level,Very High,Excellent,Medium,High,Low,Bad
1,49,No,Travel_Frequently,279,Research & Development,8,Life Sciences,1,2,Male,...,Medium,Below College,Low,Junior Level,Medium,Outstanding,High,Medium,Very High,Better
2,37,Yes,Travel_Rarely,1373,Research & Development,2,Other,1,4,Male,...,Low,College,No Options,Entry Level,High,Excellent,Very High,Medium,Medium,Better
3,33,No,Travel_Frequently,1392,Research & Development,3,Life Sciences,1,5,Female,...,Low,Master,No Options,Entry Level,High,Excellent,Very High,High,High,Better
4,27,No,Travel_Rarely,591,Research & Development,2,Medical,1,7,Male,...,Medium,Below College,Low,Entry Level,Medium,Excellent,Low,High,Very High,Better


In [17]:
all_logistic_models_dict_c = {
    **categorical_models_dict_c,
    **mixed_models_dict_c
}

len(all_logistic_models_dict_c)

15

In [18]:
df_train, df_test = split_train_test_df(
    df=df,
    target=target,
    test_size=0.30,
    random_state=42
)

print("Train shape:", df_train.shape)
print("Test shape:", df_test.shape)

print("\nTrain target distribution:")
display(df_train[target].value_counts(normalize=True).round(3))

print("\nTest target distribution:")
display(df_test[target].value_counts(normalize=True).round(3))

Train shape: (1029, 43)
Test shape: (441, 43)

Train target distribution:


AttritionFlag
0    0.839
1    0.161
Name: proportion, dtype: float64


Test target distribution:


AttritionFlag
0    0.839
1    0.161
Name: proportion, dtype: float64

In [19]:
cv_comparison = run_logistic_cross_validation(
    df=df_train,
    models_dict=all_logistic_models_dict_c,
    estimators_dict=estimators_dict,
    target=target,
    n_splits=10,
    random_state=42,
    scale_numeric=True
)

cv_comparison_sorted = (
    cv_comparison
    .sort_values(
        by=["F1_Mean", "Recall_Mean", "AUC_Mean"],
        ascending=False
    )
    .reset_index(drop=True)
)

cv_comparison_sorted

,Variable_Set,Model,Accuracy_Mean,Accuracy_Std,Precision_Mean,Precision_Std,Recall_Mean,Recall_Std,F1_Mean,F1_Std,AUC_Mean,AUC_Std,N_Numeric_Variables,N_Categorical_Variables,N_Features_After_Dummies
0,Modelo 8 — Integrado Multidimensional,Logistic Regression Balanced,0.783,0.038,0.410,0.061,0.758,0.107,0.531,0.071,0.847,0.069,7,11,43
1,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.880,0.022,0.738,0.138,0.422,0.112,0.526,0.105,0.856,0.067,7,11,43
2,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression Balanced,0.748,0.027,0.360,0.032,0.716,0.102,0.478,0.044,0.813,0.068,3,8,25
3,Modelo 4 — Trajetória Organizacional,Logistic Regression Balanced,0.744,0.048,0.358,0.064,0.721,0.140,0.477,0.083,0.796,0.074,0,8,22
4,Modelo 5 — Antiguidade Organizacional,Logistic Regression Balanced,0.726,0.044,0.342,0.046,0.733,0.102,0.464,0.053,0.783,0.071,5,6,20
5,Modelo 6 — Perfil Pessoal,Logistic Regression Balanced,0.741,0.029,0.348,0.049,0.704,0.152,0.464,0.075,0.790,0.073,0,9,24
6,Modelo 4 — Experiência Profissional,Logistic Regression Balanced,0.729,0.042,0.338,0.065,0.721,0.163,0.459,0.092,0.791,0.075,4,6,19
7,Modelo 5 — Estabilidade e Benefícios,Logistic Regression Balanced,0.734,0.054,0.347,0.057,0.692,0.100,0.459,0.059,0.795,0.066,0,9,24
8,Modelo 6 — Perfil Pessoal e Condições de Trabalho,Logistic Regression Balanced,0.735,0.028,0.341,0.043,0.703,0.132,0.458,0.066,0.791,0.072,3,8,24
9,Modelo 1 — Função Profissional,Logistic Regression Balanced,0.726,0.058,0.339,0.077,0.697,0.109,0.455,0.089,0.789,0.056,0,8,26


In [20]:
gap_analysis = run_logistic_gap_analysis(
    df=df_train,
    models_dict=all_logistic_models_dict_c,
    estimators_dict=estimators_dict,
    target=target,
    n_splits=10,
    random_state=42,
    scale_numeric=True
)

gap_analysis_sorted = (
    gap_analysis
    .sort_values(
        by=["CV_F1_Mean", "CV_Recall_Mean", "CV_AUC_Mean"],
        ascending=False
    )
    .reset_index(drop=True)
)

gap_analysis_sorted

,Variable_Set,Model,Train_F1_Mean,CV_F1_Mean,F1_Gap,Train_Recall_Mean,CV_Recall_Mean,Recall_Gap,Train_AUC_Mean,CV_AUC_Mean,AUC_Gap,Gap_Diagnosis,N_Numeric_Variables,N_Categorical_Variables,N_Features_After_Dummies
0,Modelo 8 — Integrado Multidimensional,Logistic Regression Balanced,0.571,0.531,0.041,0.823,0.758,0.065,0.893,0.847,0.046,Stable generalization,7,11,43
1,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.606,0.526,0.080,0.483,0.422,0.062,0.890,0.856,0.033,Moderate gap,7,11,43
2,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression Balanced,0.515,0.478,0.037,0.780,0.716,0.064,0.840,0.813,0.027,Stable generalization,3,8,25
3,Modelo 4 — Trajetória Organizacional,Logistic Regression Balanced,0.507,0.477,0.030,0.762,0.721,0.041,0.827,0.796,0.031,Stable generalization,0,8,22
4,Modelo 5 — Antiguidade Organizacional,Logistic Regression Balanced,0.480,0.464,0.016,0.760,0.733,0.027,0.811,0.783,0.028,Stable generalization,5,6,20
5,Modelo 6 — Perfil Pessoal,Logistic Regression Balanced,0.504,0.464,0.040,0.757,0.704,0.053,0.825,0.790,0.035,Stable generalization,0,9,24
6,Modelo 4 — Experiência Profissional,Logistic Regression Balanced,0.492,0.459,0.034,0.765,0.721,0.044,0.814,0.791,0.023,Stable generalization,4,6,19
7,Modelo 5 — Estabilidade e Benefícios,Logistic Regression Balanced,0.508,0.459,0.049,0.762,0.692,0.070,0.832,0.795,0.037,Stable generalization,0,9,24
8,Modelo 6 — Perfil Pessoal e Condições de Trabalho,Logistic Regression Balanced,0.500,0.458,0.041,0.756,0.703,0.053,0.828,0.791,0.037,Stable generalization,3,8,24
9,Modelo 1 — Função Profissional,Logistic Regression Balanced,0.482,0.455,0.027,0.740,0.697,0.043,0.831,0.789,0.042,Stable generalization,0,8,26


In [21]:
general_results, threshold_results, confusion_results, trained_models, interpretation_results = run_logistic_model_comparison(
    df=df_train,
    models_dict=all_logistic_models_dict_c,
    estimators_dict=estimators_dict,
    target=target,
    thresholds=np.arange(0.20, 0.651, 0.025),
    test_size=0.30,
    random_state=42,
    scale_numeric=True
)

general_results_sorted = (
    general_results
    .sort_values(
        by=["F1-score", "Recall", "AUC"],
        ascending=False
    )
    .reset_index(drop=True)
)

general_results_sorted

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC,N_Numeric_Variables,N_Categorical_Variables,N_Features_After_Dummies
0,Modelo 8 — Integrado Multidimensional,Logistic Regression Balanced,0.5,0.809,0.446,0.74,0.556,0.823,7,11,43
1,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression Balanced,0.5,0.764,0.384,0.76,0.510,0.808,3,8,25
2,Modelo 6 — Perfil Pessoal e Condições de Trabalho,Logistic Regression Balanced,0.5,0.751,0.361,0.70,0.476,0.786,3,8,24
3,Modelo 4 — Trajetória Organizacional,Logistic Regression Balanced,0.5,0.741,0.347,0.68,0.459,0.782,0,8,22
4,Modelo 6 — Perfil Pessoal,Logistic Regression Balanced,0.5,0.741,0.347,0.68,0.459,0.774,0,9,24
5,Modelo 1 — Função Profissional Misto,Logistic Regression Balanced,0.5,0.751,0.348,0.62,0.446,0.779,3,7,26
6,Modelo 1 — Função Profissional,Logistic Regression Balanced,0.5,0.741,0.340,0.64,0.444,0.770,0,8,26
7,Modelo 4 — Experiência Profissional,Logistic Regression Balanced,0.5,0.722,0.327,0.68,0.442,0.771,4,6,19
8,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.5,0.861,0.630,0.34,0.442,0.834,7,11,43
9,Modelo 2 — Nível Hierárquico,Logistic Regression Balanced,0.5,0.709,0.318,0.70,0.438,0.768,0,8,22


In [22]:
top_combinations = (
    cv_comparison_sorted
    .head(20)[["Variable_Set", "Model"]]
    .to_dict("records")
)

top_combinations

[{'Variable_Set': 'Modelo 8 — Integrado Multidimensional',
  'Model': 'Logistic Regression Balanced'},
 {'Variable_Set': 'Modelo 8 — Integrado Multidimensional',
  'Model': 'Logistic Regression'},
 {'Variable_Set': 'Modelo 2 — Nível Hierárquico e Benefícios',
  'Model': 'Logistic Regression Balanced'},
 {'Variable_Set': 'Modelo 4 — Trajetória Organizacional',
  'Model': 'Logistic Regression Balanced'},
 {'Variable_Set': 'Modelo 5 — Antiguidade Organizacional',
  'Model': 'Logistic Regression Balanced'},
 {'Variable_Set': 'Modelo 6 — Perfil Pessoal',
  'Model': 'Logistic Regression Balanced'},
 {'Variable_Set': 'Modelo 4 — Experiência Profissional',
  'Model': 'Logistic Regression Balanced'},
 {'Variable_Set': 'Modelo 5 — Estabilidade e Benefícios',
  'Model': 'Logistic Regression Balanced'},
 {'Variable_Set': 'Modelo 6 — Perfil Pessoal e Condições de Trabalho',
  'Model': 'Logistic Regression Balanced'},
 {'Variable_Set': 'Modelo 1 — Função Profissional',
  'Model': 'Logistic Regressio

In [23]:
tuning_results_df, best_models = tune_logistic_hyperparameters_top_combinations(
    df=df_train,
    models_dict=all_logistic_models_dict_c,
    estimators_dict=estimators_dict,
    param_distributions_dict=param_distributions_dict,
    top_combinations=top_combinations,
    target=target,
    n_iter=30,
    n_splits=10,
    scoring="f1",
    random_state=42,
    scale_numeric=True
)

tuning_results_sorted = (
    tuning_results_df
    .sort_values(
        by="Best_Score",
        ascending=False
    )
    .reset_index(drop=True)
)

tuning_results_sorted

/Users/nathansperinde/Desktop/Portifolio/projeto_1/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/nathansperinde/Desktop/Portifolio/projeto_1/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/Users/nathansperinde/Desktop/Portifolio/projeto_1/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning

,Variable_Set,Model,Best_Score,Scoring,Best_Params,N_Parameter_Combinations_Tested,N_Numeric_Variables,N_Categorical_Variables,N_Features_After_Dummies
0,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.542,f1,"{'classifier__solver': 'liblinear', 'classifie...",10,7,11,43
1,Modelo 8 — Integrado Multidimensional,Logistic Regression Balanced,0.531,f1,"{'classifier__solver': 'liblinear', 'classifie...",10,7,11,43
2,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression Balanced,0.484,f1,"{'classifier__solver': 'liblinear', 'classifie...",10,3,8,25
3,Modelo 4 — Trajetória Organizacional,Logistic Regression Balanced,0.482,f1,"{'classifier__solver': 'liblinear', 'classifie...",10,0,8,22
4,Modelo 6 — Perfil Pessoal,Logistic Regression Balanced,0.470,f1,"{'classifier__solver': 'liblinear', 'classifie...",10,0,9,24
5,Modelo 4 — Experiência Profissional,Logistic Regression Balanced,0.467,f1,"{'classifier__solver': 'liblinear', 'classifie...",10,4,6,19
6,Modelo 5 — Antiguidade Organizacional,Logistic Regression Balanced,0.465,f1,"{'classifier__solver': 'liblinear', 'classifie...",10,5,6,20
7,Modelo 5 — Estabilidade e Benefícios,Logistic Regression Balanced,0.465,f1,"{'classifier__solver': 'liblinear', 'classifie...",10,0,9,24
8,Modelo 6 — Perfil Pessoal e Condições de Trabalho,Logistic Regression Balanced,0.465,f1,"{'classifier__solver': 'liblinear', 'classifie...",10,3,8,24
9,Modelo 1 — Função Profissional,Logistic Regression Balanced,0.460,f1,"{'classifier__solver': 'liblinear', 'classifie...",10,0,8,26


In [24]:
(
    threshold_cv_comparison,
    best_thresholds_cv,
    final_test_results_df,
    confusion_results_opt,
    fitted_models,
    interpretation_results_opt
) = evaluate_thresholds_optimized_logistic_models_cv(
    df=df_train,
    df_test=df_test,
    models_dict=all_logistic_models_dict_c,
    best_models=best_models,
    target=target,
    thresholds=np.arange(0.20, 0.751, 0.01),
    test_size=0.30,
    random_state=42,
    n_splits=10,
    threshold_metric="F1-score"
)

best_thresholds_cv

/Users/nathansperinde/Desktop/Portifolio/projeto_1/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/nathansperinde/Desktop/Portifolio/projeto_1/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penal

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC
1,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.38,0.874,0.615,0.578,0.596,0.853
0,Modelo 8 — Integrado Multidimensional,Logistic Regression Balanced,0.64,0.846,0.518,0.681,0.589,0.842
16,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression,0.33,0.848,0.530,0.524,0.527,0.812
2,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression Balanced,0.71,0.848,0.532,0.506,0.519,0.807
8,Modelo 6 — Perfil Pessoal e Condições de Trabalho,Logistic Regression Balanced,0.59,0.801,0.424,0.651,0.513,0.790
17,Modelo 5 — Estabilidade e Benefícios,Logistic Regression,0.23,0.807,0.430,0.608,0.504,0.798
7,Modelo 5 — Estabilidade e Benefícios,Logistic Regression Balanced,0.74,0.856,0.568,0.452,0.503,0.791
3,Modelo 4 — Trajetória Organizacional,Logistic Regression Balanced,0.57,0.791,0.408,0.651,0.501,0.792
18,Modelo 4 — Trajetória Organizacional,Logistic Regression,0.26,0.824,0.462,0.542,0.499,0.795
5,Modelo 6 — Perfil Pessoal,Logistic Regression Balanced,0.56,0.780,0.394,0.675,0.498,0.786


In [25]:
final_test_results_sorted = (
    final_test_results_df
    .sort_values(
        by=["F1-score", "Recall", "AUC"],
        ascending=False
    )
    .reset_index(drop=True)
)

final_test_results_sorted

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC
0,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.38,0.864,0.575,0.592,0.583,0.814
1,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression Balanced,0.71,0.864,0.582,0.549,0.565,0.830
2,Modelo 8 — Integrado Multidimensional,Logistic Regression Balanced,0.64,0.837,0.495,0.634,0.556,0.810
3,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression,0.33,0.853,0.543,0.535,0.539,0.833
4,Modelo 3 — Faixa Salarial,Logistic Regression Balanced,0.58,0.810,0.437,0.634,0.517,0.797
5,Modelo 2 — Nível Hierárquico,Logistic Regression Balanced,0.62,0.823,0.461,0.577,0.512,0.817
6,Modelo 6 — Perfil Pessoal e Condições de Trabalho,Logistic Regression Balanced,0.59,0.816,0.447,0.592,0.509,0.802
7,Modelo 4 — Experiência Profissional,Logistic Regression Balanced,0.61,0.821,0.456,0.577,0.509,0.796
8,Modelo 5 — Antiguidade Organizacional,Logistic Regression Balanced,0.60,0.807,0.430,0.606,0.503,0.781
9,Modelo 6 — Perfil Pessoal,Logistic Regression Balanced,0.56,0.794,0.407,0.620,0.492,0.802


In [26]:
final_summary = (
    final_test_results_sorted
    .merge(
        tuning_results_sorted[
            ["Variable_Set", "Model", "Best_Score", "Best_Params"]
        ],
        on=["Variable_Set", "Model"],
        how="left"
    )
    .merge(
        gap_analysis_sorted[
            [
                "Variable_Set",
                "Model",
                "Train_F1_Mean",
                "CV_F1_Mean",
                "F1_Gap",
                "Train_Recall_Mean",
                "CV_Recall_Mean",
                "Recall_Gap",
                "Gap_Diagnosis"
            ]
        ],
        on=["Variable_Set", "Model"],
        how="left"
    )
)

final_summary

,Variable_Set,Model,Threshold,Accuracy,Precision,Recall,F1-score,AUC,Best_Score,Best_Params,Train_F1_Mean,CV_F1_Mean,F1_Gap,Train_Recall_Mean,CV_Recall_Mean,Recall_Gap,Gap_Diagnosis
0,Modelo 8 — Integrado Multidimensional,Logistic Regression,0.38,0.864,0.575,0.592,0.583,0.814,0.542,"{'classifier__solver': 'liblinear', 'classifie...",0.606,0.526,0.080,0.483,0.422,0.062,Moderate gap
1,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression Balanced,0.71,0.864,0.582,0.549,0.565,0.830,0.484,"{'classifier__solver': 'liblinear', 'classifie...",0.515,0.478,0.037,0.780,0.716,0.064,Stable generalization
2,Modelo 8 — Integrado Multidimensional,Logistic Regression Balanced,0.64,0.837,0.495,0.634,0.556,0.810,0.531,"{'classifier__solver': 'liblinear', 'classifie...",0.571,0.531,0.041,0.823,0.758,0.065,Stable generalization
3,Modelo 2 — Nível Hierárquico e Benefícios,Logistic Regression,0.33,0.853,0.543,0.535,0.539,0.833,0.405,"{'classifier__solver': 'liblinear', 'classifie...",0.420,0.388,0.032,0.292,0.271,0.021,Stable generalization
4,Modelo 3 — Faixa Salarial,Logistic Regression Balanced,0.58,0.810,0.437,0.634,0.517,0.797,0.442,"{'classifier__solver': 'liblinear', 'classifie...",0.475,0.440,0.035,0.731,0.679,0.052,Stable generalization
5,Modelo 2 — Nível Hierárquico,Logistic Regression Balanced,0.62,0.823,0.461,0.577,0.512,0.817,0.451,"{'classifier__solver': 'liblinear', 'classifie...",0.468,0.448,0.020,0.722,0.692,0.031,Stable generalization
6,Modelo 6 — Perfil Pessoal e Condições de Trabalho,Logistic Regression Balanced,0.59,0.816,0.447,0.592,0.509,0.802,0.465,"{'classifier__solver': 'liblinear', 'classifie...",0.500,0.458,0.041,0.756,0.703,0.053,Stable generalization
7,Modelo 4 — Experiência Profissional,Logistic Regression Balanced,0.61,0.821,0.456,0.577,0.509,0.796,0.467,"{'classifier__solver': 'liblinear', 'classifie...",0.492,0.459,0.034,0.765,0.721,0.044,Stable generalization
8,Modelo 5 — Antiguidade Organizacional,Logistic Regression Balanced,0.60,0.807,0.430,0.606,0.503,0.781,0.465,"{'classifier__solver': 'liblinear', 'classifie...",0.480,0.464,0.016,0.760,0.733,0.027,Stable generalization
9,Modelo 6 — Perfil Pessoal,Logistic Regression Balanced,0.56,0.794,0.407,0.620,0.492,0.802,0.470,"{'classifier__solver': 'liblinear', 'classifie...",0.504,0.464,0.040,0.757,0.704,0.053,Stable generalization


In [48]:
for i in range(0, 20):

    best_models = final_summary.iloc[i]

    if best_models["Variable_Set"] == "Modelo 8 — Integrado Multidimensional":
        continue

    print("Variable set:", best_models["Variable_Set"])
    print("Model:", best_models["Model"])
    print("Best threshold:", best_models["Threshold"])

    confusion_mat = confusion_results_opt[
        (best_models["Variable_Set"], best_models["Model"])
    ]

    interpretation = interpretation_results_opt[
        (best_models["Variable_Set"], best_models["Model"])
    ]

    top_positive_odds = (
        interpretation
        .sort_values("Odds_Ratio", ascending=False)
        .head(20)
        .reset_index(drop=True)
    )

    top_negative_odds = (
        interpretation
        .sort_values("Odds_Ratio", ascending=True)
        .head(20)
        .reset_index(drop=True)
    )

    display(best_models)

    display(interpretation)

    display(confusion_mat)

    print("\n\n")

Variable set: Modelo 2 — Nível Hierárquico e Benefícios
Model: Logistic Regression Balanced
Best threshold: 0.71


Variable_Set                 Modelo 2 — Nível Hierárquico e Benefícios
Model                                     Logistic Regression Balanced
Threshold                                                         0.71
Accuracy                                                         0.864
Precision                                                        0.582
Recall                                                           0.549
F1-score                                                         0.565
AUC                                                               0.83
Best_Score                                                       0.484
Best_Params          {'classifier__solver': 'liblinear', 'classifie...
Train_F1_Mean                                                    0.515
CV_F1_Mean                                                       0.478
F1_Gap                                                           0.037
Train_Recall_Mean                                                 0.78
CV_Rec

,Feature,Coefficient,Odds_Ratio
0,BusinessTravel_Travel_Frequently,1.519700,4.570854
1,OverTime_Yes,1.448745,4.257766
2,JobInvolvementLevel_Low,1.406853,4.083084
3,BusinessTravel_Travel_Rarely,0.837211,2.309917
4,EnvironmentSatisfactionLevel_Low,0.785172,2.192784
5,StockOption_No Options,0.549700,1.732732
6,SatisfactionLevel_Low,0.413862,1.512649
7,SatisfactionLevel_Medium,0.304365,1.355763
8,DistanceFromHome,0.281086,1.324568
9,JobInvolvementLevel_Medium,0.061730,1.063675


array([[342,  28],
       [ 32,  39]])




Variable set: Modelo 2 — Nível Hierárquico e Benefícios
Model: Logistic Regression
Best threshold: 0.33


...

array([[278,  92],
       [ 25,  46]])




Variable set: Modelo 4 — Trajetória Organizacional
Model: Logistic Regression Balanced
Best threshold: 0.57


Variable_Set                      Modelo 4 — Trajetória Organizacional
Model                                     Logistic Regression Balanced
Threshold                                                         0.57
Accuracy                                                         0.766
Precision                                                        0.349
Recall                                                           0.521
F1-score                                                         0.418
AUC                                                              0.771
Best_Score                                                       0.482
Best_Params          {'classifier__solver': 'liblinear', 'classifie...
Train_F1_Mean                                                    0.507
CV_F1_Mean                                                       0.477
F1_Gap                                                            0.03
Train_Recall_Mean                                                0.762
CV_Rec

,Feature,Coefficient,Odds_Ratio
0,CareerStage_Entry-level,2.309079,10.065152
1,OverTime_Yes,1.461711,4.313335
2,JobInvolvementLevel_Low,1.389260,4.011880
3,BusinessTravel_Travel_Frequently,1.301969,3.676529
4,EnvironmentSatisfactionLevel_Low,0.900192,2.460074
5,BusinessTravel_Travel_Rarely,0.616466,1.852371
6,SatisfactionLevel_Low,0.496854,1.643543
7,DistanceGroup_21-30,0.487436,1.628137
8,DistanceGroup_11-20,0.373423,1.452699
9,JobInvolvementLevel_Medium,0.181397,1.198891


array([[301,  69],
       [ 34,  37]])